In [0]:
%run ../utils/init.py

In [0]:
files = dbutils.fs.ls(f"{RAW}/dpe/dept=75/")
for f in files:
    print(f.name, f.size)

In [0]:
df = spark.read.parquet(f"{RAW}/dpe/dept=75/")
df.printSchema()
df.show(2)

In [0]:
df_raw = spark.read.parquet(f"{RAW}/dpe/")
print(f"Lignes brutes : {df_raw.count()}")
df_raw.show(2)

In [0]:
%pip install pyproj

In [0]:
import pandas as pd
from pyproj import Transformer
from pyspark.sql.types import DoubleType
from pyspark.sql.functions import current_timestamp, to_date, year

# UDF reprojection Lambert93 → WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)

@F.udf(DoubleType())
def to_lon_wgs84(x, y):
    if x is None or y is None:
        return None
    lon, lat = transformer.transform(x, y)
    return float(lon)

@F.udf(DoubleType())
def to_lat_wgs84(x, y):
    if x is None or y is None:
        return None
    lon, lat = transformer.transform(x, y)
    return float(lat)

# Mapping étiquette → score numérique
dpe_mapping = F.create_map(
    F.lit("A"), F.lit(1), F.lit("B"), F.lit(2),
    F.lit("C"), F.lit(3), F.lit("D"), F.lit(4),
    F.lit("E"), F.lit(5), F.lit("F"), F.lit(6),
    F.lit("G"), F.lit(7)
)

df_bronze = (df_raw
    .filter(F.col("etiquette_dpe").isin(["A","B","C","D","E","F","G"]))
    .filter(F.col("surface_m2").isNull() | F.col("surface_m2").between(9, 2000))
    .withColumn("date_dpe", to_date("date_dpe", "yyyy-MM-dd"))
    .withColumn("annee_dpe", year("date_dpe"))
    .withColumn("score_dpe_num", dpe_mapping[F.trim(F.upper("etiquette_dpe"))])
    .withColumn("lon_wgs84", to_lon_wgs84(F.col("longitude"), F.col("latitude")))
    .withColumn("lat_wgs84", to_lat_wgs84(F.col("longitude"), F.col("latitude")))
    .drop("longitude", "latitude", "dept", "_score")
    .withColumnRenamed("lon_wgs84", "longitude")
    .withColumnRenamed("lat_wgs84", "latitude")
    .withColumn("ingestion_timestamp", current_timestamp())
)

print(f"Lignes après filtres : {df_bronze.count()}")

In [0]:
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("annee_dpe", "code_departement") \
    .save(f"{BRONZE}/dpe/")

print("✅ Bronze DPE écrit")